<a href="https://colab.research.google.com/github/MitraShabani/Merge-Order-Bias/blob/main/merge_order_stability_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai sentence-transformers matplotlib numpy

In [ ]:
import json
import re
from sentence_transformers import SentenceTransformer, util
import matplotlib.pyplot as plt

embedder = SentenceTransformer('all-MiniLM-L6-v2') # measures the angle between two of these coordinate vectors.
print("Embedding model loaded.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Loading the results
import os

RESULTS_DIR = "/content/drive/MyDrive/merge-order-bias/results"
PLOTS_DIR = "/content/drive/MyDrive/merge-order-bias/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

all_files = os.listdir(RESULTS_DIR)

# A book is "complete" if both its forward and backward files exist
book_slugs = set(f.replace("_forward.json", "").replace("_backward.json", "")
                  for f in all_files if f.endswith(".json"))

book_pairs = []
for slug in book_slugs:
    has_forward = f"{slug}_forward.json" in all_files
    has_backward = f"{slug}_backward.json" in all_files
    if has_forward and has_backward:
        book_pairs.append(slug)
    else:
        print(f"WARNING: {slug} is missing forward or backward results — skipping")

print(f"Found {len(book_pairs)} complete book pairs: {book_pairs}")

In [ ]:
# Extract each chapter's FIRST and LAST appearance text
def get_first_appearance_text(merge_log, chapter_num):
    """Return the full summary text from the step where `chapter_num` was first introduced."""
    for step in merge_log:
        if chapter_num in step["chapters_included"]:
            return step["summary"]
    return None


def get_last_appearance_text(merge_log, chapter_num):
    """Return the full summary text from the LAST step that includes `chapter_num`
    (in this design, that's simply the final step, since chapters are never dropped)."""
    for step in reversed(merge_log):
        if chapter_num in step["chapters_included"]:
            return step["summary"]
    return None


print("Helper functions ready.")

In [ ]:
# Compute drift: Compute drift per chapter, per direction

""" For each chapter: compare its original standalone summary to (a) the step where it was first introduced, and (b) the FINAL summary as a whole.
The gap between (a) and (b) tells us how much that chapter's presence/similarity degraded from its first appearance to the end of the merge process."""

def compute_drift(merge_log, num_chapters):
    results = []
    for i in range(num_chapters):
        chapter_num = i + 1
        first_text = get_first_appearance_text(merge_log, chapter_num)
        last_text = get_last_appearance_text(merge_log, chapter_num)

        emb_first = embedder.encode(first_text, convert_to_tensor=True)
        emb_last = embedder.encode(last_text, convert_to_tensor=True)

        stability = util.cos_sim(emb_first, emb_last).item()

        results.append({
            "chapter": chapter_num,
            "stability": stability  # HIGH = barely changed from first appearance to the end (dominates/persists)
                                     # LOW = changed a lot (got compressed/diluted/rewritten heavily)
        })
    return results

summary_data = []
for book_slug in book_pairs:
    with open(os.path.join(RESULTS_DIR, f"{book_slug}_forward.json"), "r", encoding="utf-8") as f:
        forward = json.load(f)
    with open(os.path.join(RESULTS_DIR, f"{book_slug}_backward.json"), "r", encoding="utf-8") as f:
        backward = json.load(f)

    book_title = forward.get("book_title", book_slug)
    num_chapters = forward["num_chapters"]
    print(f"\n=== {book_title} ({num_chapters} chapters) ===")

    forward_drift = compute_drift(forward["merge_log"], num_chapters)
    backward_drift = compute_drift(backward["merge_log"], num_chapters)

    # Plot: stability (first-appearance vs. final) by chapter position,, forward vs. backward
    chapters = [d["chapter"] for d in forward_drift]
    forward_stability = [d["stability"] for d in forward_drift]
    backward_stability = [d["stability"] for d in backward_drift]

    plt.figure(figsize=(10, 6))
    plt.plot(chapters, forward_stability, marker='o', label='Forward merge', color='steelblue')
    plt.plot(chapters, backward_stability, marker='s', label='Backward merge', color='firebrick')
    plt.xlabel("Chapter number (book order)")
    plt.ylabel("Stability: first-appearance vs. final-step similarity")
    plt.title(f"{book_title}: stability by chapter, forward vs. backward")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(chapters)
    plot_path = os.path.join(PLOTS_DIR, f"stability_{book_slug}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved plot: {plot_path}")

    # Record the key comparable value for the cross-book summary

    forward_first_chapter_stability = forward_drift[0]["stability"]  # chapter 1
    backward_first_chapter_stability = backward_drift[-1]["stability"]  # last chapter

    summary_data.append({
            "book": book_title,
            "num_chapters": num_chapters,
            "forward_first_chapter_stability": forward_first_chapter_stability,
            "backward_first_chapter_stability": backward_first_chapter_stability
        })

print("\n=== All books processed ===")

